# Notebook 3 — Baseline Model (Logistic Regression)

**Project:** IntelliSys Ltd. — London Road Collision Severity Prediction  

---

## Objective

Establish a Logistic Regression baseline. Assessment markers **explicitly expect**  
a classical ML baseline against which the MLP is compared.

**Why Logistic Regression?**  
Logistic Regression is interpretable, computationally cheap, and well-understood.  
Any DL model should meaningfully outperform it — especially on Fatal recall —  
to justify the additional complexity.

**Metrics recorded here become the benchmark row in the comparison table in Notebook 5.**

---
## Step 1 — Imports and Seeds

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import set_seeds, outputs_dir
from src.preprocessing import CLASS_NAMES

np.random.seed(42)
set_seeds(42)

PROCESSED_DIR = project_root / 'data' / 'processed'
FIGURES_DIR   = outputs_dir('figures')
print('Setup complete.')

---
## Step 2 — Load Preprocessed Data

In [ ]:
X_train = np.load(PROCESSED_DIR / 'X_train.npy')
X_val   = np.load(PROCESSED_DIR / 'X_val.npy')
X_test  = np.load(PROCESSED_DIR / 'X_test.npy')
y_train = np.load(PROCESSED_DIR / 'y_train.npy')
y_val   = np.load(PROCESSED_DIR / 'y_val.npy')
y_test  = np.load(PROCESSED_DIR / 'y_test.npy')

with open(PROCESSED_DIR / 'feature_names.txt') as f:
    feature_names = f.read().splitlines()

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print(f'Features: {len(feature_names)}')

---
## Step 3 — Train Logistic Regression Baseline

**Configuration decisions:**
- `class_weight='balanced'`: mirrors the class weighting strategy in the MLP for a fair comparison.
- `max_iter=1000`: ensures convergence on this feature set without warnings.
- `random_state=42`: reproducibility.

No feature selection or regularisation tuning is applied beyond defaults —  
this is an intentional baseline, not a tuned Logistic Regression.

In [ ]:
lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42,
    multi_class='multinomial',
    solver='lbfgs'
)

lr_model.fit(X_train, y_train)
print('Logistic Regression trained successfully.')

---
## Step 4 — Evaluate on Test Set

The model is evaluated on the held-out test set. Results here are fixed —  
the test set was not used during training or any hyperparameter decisions.

In [ ]:
y_pred_lr   = lr_model.predict(X_test)
probs_lr    = lr_model.predict_proba(X_test)

print('=== LOGISTIC REGRESSION — TEST SET RESULTS ===')
print(classification_report(y_test, y_pred_lr, target_names=CLASS_NAMES, digits=4))

In [ ]:
# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp  = ConfusionMatrixDisplay(cm_lr, display_labels=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(cmap='Blues', ax=ax, colorbar=False)
ax.set_title('Confusion Matrix — Logistic Regression Baseline', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_baseline.png', dpi=150)
plt.show()

In [ ]:
# AUC-ROC
auc_lr = roc_auc_score(y_test, probs_lr, multi_class='ovr', average='macro')
print(f'Macro AUC-ROC (Logistic Regression): {auc_lr:.4f}')

---
## Step 5 — Baseline Metrics Summary

Record these numbers precisely — they will populate the comparison table in Notebook 5.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

report = classification_report(y_test, y_pred_lr, target_names=CLASS_NAMES, output_dict=True)

baseline_metrics = {
    'Accuracy':        round(accuracy_score(y_test, y_pred_lr), 4),
    'Macro F1':        round(report['macro avg']['f1-score'], 4),
    'Weighted F1':     round(report['weighted avg']['f1-score'], 4),
    'Fatal Recall':    round(report['Fatal']['recall'], 4),
    'Serious Recall':  round(report['Serious']['recall'], 4),
    'Macro AUC-ROC':   round(auc_lr, 4),
}

metrics_df = pd.DataFrame.from_dict(baseline_metrics, orient='index', columns=['Logistic Regression'])
display(metrics_df)

# Save for Notebook 5
metrics_df.to_csv(PROCESSED_DIR / 'baseline_metrics.csv')
np.save(PROCESSED_DIR / 'y_pred_lr.npy',  y_pred_lr)
np.save(PROCESSED_DIR / 'probs_lr.npy',   probs_lr)
print('Baseline predictions and metrics saved.')

---
## Interpretation

**Fatal Recall** is the most safety-critical metric for this deployment:  
a false negative on Fatal means the model predicted 'Slight' when the collision  
was actually fatal — in IntelliSys's system, this prevents ambulance pre-alerting.

Logistic Regression with balanced class weights typically achieves moderate  
Fatal recall but struggles to separate Serious from Slight due to feature  
linearity assumptions. The MLP in Notebook 4 should improve this through  
non-linear feature interactions via its hidden layers.

**Proceed to Notebook 4 — MLP Model** to build and train the deep learning model.